# **3.4 Softmax Regression**
Typically, machine learning practitioners use the term “classification” to describe two subtly different problems: **1. We are only interested in the “hard” categories of the samples—that is, which category each sample belongs to; 2. We want to obtain “soft” categories—that is, the probability that each sample belongs to each category.** The line between these two is often blurred. **One reason for this is that even when we are only concerned with hard categories, we still use models designed for soft categories.**

## **3.4.1 Classification Problems**
This is a general classification problem that does not involve any natural order among the categories. Fortunately, statisticians devised a simple method for representing classification data long ago: one-hot encoding. One-hot encoding is a vector with as many components as there are categories. The component corresponding to a category is set to 1, while all other components are set to 0. In our example, the label y would be a three-dimensional vector, where (1, 0, 0) corresponds to “cat,” (0, 1, 0) corresponds to “chicken,” and (0, 0, 1) corresponds to “dog”:
##### **(3.4.1)**
$$y \in {(1,0,0),(0,1,0),(0,0,1)}$$

## **3.4.3 Network Architecture**
**To estimate the conditional probabilities for all possible classes, we need a model with multiple outputs, with one output corresponding to each class.** To solve a classification problem using a linear model, we need as many affine functions as there are outputs. Each output corresponds to its own affine function. In our example, since we have 4 features and 3 possible output categories, we will need 12 scalars to represent the weights (w with subscripts) and 3 scalars to represent the biases (b with subscripts). Below, we calculate three unnormalized predictions (logits) for each input: o1, o2, and o3.
##### **(3.4.2)**
$$
\begin{align}
o_1=x_1w_{11}+x_2w_{12}+x_3w_{13}+x_4w_{14}+b_1, \\
o_2=x_1w_{21}+x_2w_{22}+x_3w_{23}+x_4w_{24}+b_2, \\
o_3=x_1w_{31}+x_2w_{32}+x_3w_{33}+x_4w_{34}+b_3. \\
\end{align}
$$
Like linear regression, softmax regression is also a single-layer neural network. **Since the calculation of each output o1, o2, and o3 depends on all inputs x1, x2, x3, and x4, the output layer of softmax regression is also a fully connected layer.**
##### **fig(3.4.1)**
![illustration of softmax regression](../../images/picture3_4_1.png)<br><br>
Expressed in vector form as $o = Wx + b$, this is a format better suited for mathematics and coding. Thus, we have placed all the weights into a 3×4 matrix. For a given data sample with features $x$, our output is obtained by performing matrix-vector multiplication of the weights with the input features, plus the bias $b$.

## **3.4.3 Parameter Overhead in Fully Connected Layers**
As the name suggests, a fully connected layer is “fully” connected and may have a large number of trainable parameters. Specifically, for any fully connected layer with $d$ inputs and $q$ outputs, the parameter overhead is $\mathcal{O}(dq)$, a figure that can be prohibitively high in practice. Fortunately, the cost of mapping $d$ inputs to $q$ outputs can be reduced to $\mathcal{O}\left(\frac{dq}{n}\right)$, where the hyperparameter n can be flexibly specified to balance parameter efficiency and model effectiveness in practical applications.

## **3.4.4 Softmax Operation**
Now we will optimize the parameters to maximize the probability of the observed data. To obtain the prediction results, we will set a threshold, such as selecting the label with the highest probability.<br><br>
We hope that the model's output $\hat{y}_j$ can be interpreted as the probability of belonging to class $j$, and then select the class with the maximum output value, $argmax_jy_j$, as our prediction.<br><br>
However, can we treat the unnormalized prediction $o$ directly as the output we are interested in? The answer is no. **This is because treating the output of a linear layer directly as a probability presents some issues: on the one hand, we have not constrained the sum of these output values to be 1. On the other hand, depending on the input, they can be negative.**<br><br>
To treat the outputs as probabilities, we must ensure that the outputs for any given dataset are non-negative and sum to 1. In addition, we need a training objective function to incentivize the model to estimate probabilities accurately. For example, among all samples for which the classifier outputs 0.5, we want exactly half of those samples to actually belong to the predicted class. This property is called calibration.<br><br>
The softmax function transforms unnormalized predictions into non-negative numbers that sum to 1, while ensuring the model remains differentiable. To achieve this, we first raise each unnormalized prediction to a power, which ensures that the output is non-negative. To ensure that the final probability values sum to 1, we then divide each of the raised-to-a-power results by their sum. As shown in the following equation:
##### **(3.4.3)**
$$\hat{y}=softmax(o),\hat{y}_{j}=\frac{exp{o_j}}{\sum_{k}exp(o_k)}$$
Here, for all j, $0 ≤ \hat{y}_j ≤ 1$. Therefore, $\hat{y}$ can be regarded as a valid probability distribution. **The softmax operation does not alter the order of magnitude among the unnormalized predictions $o$; it merely determines the probabilities assigned to each class.** Thus, during the prediction process, we can still use the following formula to select the most likely class.
##### **(3.4.4)**
$$\underset{j}{\arg\max}\hat{y}_j=\underset{j}{\arg\max}{o_{j}}$$
Although the softmax is a nonlinear function, **the output of softmax regression is still determined by an affine transformation of the input features.** Therefore, softmax regression is a linear model.

## **3.4.5 Vectorization of Small-Batch Samples**
To improve computational efficiency and make full use of the GPU, we typically perform vector computations on small batches of data. Suppose we have a batch of samples X, where the feature dimension (number of inputs) is d and the batch size is n. Furthermore, suppose there are q classes in the output. Then, the features of the batch of samples are $X \in \mathbb{R}^{n×d}$, the weights are $W \in \mathbb{R}^{d×q}$, and the bias is $b \in \mathbb{R}^{1×q}$. The vector calculation expression for softmax regression is:
##### **(3.4.5)**
$$
\begin{align}
O=XW+b, \\
\hat{Y}=softmax(O). \\
\end{align}
$$
Compared to processing one sample at a time, vectorizing small batches of samples accelerates the matrix-vector multiplication in XW. Since each row in X represents a data sample, the softmax operation can be performed rowwise: for each row of O, we first raise all terms to the power, then normalize them by summing them. In [(3.4.5)](#345), the summation over XW+b utilizes broadcasting, as both the mini-batch of unnormalized predictions O and the output probabilities $\hat{Y}$ are matrices of shape $n×q$.

## **3.4.6 Loss Function**
Next, we need a loss function to measure the performance of our predictions. We will use maximum likelihood estimation, which is the same method used in linear regression (Section [3.1.3](linear_regression.ipynb#313)).

### **Log-likelihood**
The softmax function returns a vector $\hat{y}$, which we can interpret as “the conditional probability of each class given any input $x$.” For example, $\hat{y} = P(y = cat | x)$. Suppose the entire dataset ${X, Y}$ contains n samples, where the sample at index $i$ consists of a feature vector $x^{(i)}$ and a one-hot label vector $y^{(i)}$. We can compare the estimated values with the actual values:
##### **(3.4.6)**
$$P(Y|X)=\prod\limits_{i=1}^{n}P(y^{(i)}|x^{(i)})$$
According to the maximum likelihood estimation method, we maximize $P(Y|X)$, which is equivalent to minimizing the negative log-likelihood:
##### **(3.4.7)**
$$-logP(Y|X)=\sum\limits_{i=1}^{n}-logP(y^{(i)}|x^{(i)}=\sum\limits_{i=1}^{n}l(y^{(i)},\hat{y}^{(i)})$$
In particular, for any label $y$ and model prediction $\hat{y}$, the loss function is:
##### **(3.4.8)**
$$l(y,\hat{y})=-\sum\limits_{j=1}^{q}y_{j}log\hat{y}_j$$
The loss function in [(3.4.8)](#348) is commonly referred to as cross-entropy loss. **Since $y$ is a one-hot encoded vector of length $q$, all terms $j$ except for one vanish. Since all $\hat{y}_j$ are predicted probabilities, their logarithms will never be greater than 0. Therefore, if the true label is correctly predicted—that is, if the true label is $P(y|x) = 1$—the loss function cannot be minimized any further. Note that this is often impossible. For example, there may be label noise in the dataset (such as samples that are mislabeled), or the input features may not contain enough information to perfectly classify every sample.**

### **Softmax and Its Derivatives**
Since the softmax function and the associated loss function are commonly used, we need to gain a better understanding of how they are computed. Substitute [(3.4.3)](#343) into the loss function [(3.4.8)](#348). Using the definition of the softmax function, we obtain:
##### **(3.4.9)**
$$
\begin{align}
l(y,\hat{y}) &= -\sum\limits_{j=1}^{q}log\frac{exp(o_j)}{\sum^{q}_{k=1}exp(o_k)} \\
&= -\sum\limits_{j=1}^{q}y_{j}log\sum^{q}_{k=1}exp(o_k)-\sum\limits_{j=1}^{q}y_{j}o_{j} \\
&= log\sum\limits_{j=1}^{q}exp(o_k) - \sum\limits_{j=1}^{q}y_{j}o_{j} \\
\end{align}
$$
Considering the derivative with respect to any unnormalized prediction $o_j$, we obtain:
##### **(3.4.10)**
$$\partial_{o_j}l(y,\hat{y}) = \frac{exp(o_j)}{\sum_{k=1}^{q}exp(o_k)}-y_j=softmax(o)_j-y_j$$
**In other words, the derivative represents the difference between the probability assigned by our softmax model and the actual outcome (represented by the one-hot label vector).** In this sense, it is very similar to what we see in regression, where the gradient is the difference between the observed value $y$ and the estimated value $\hat{y}$.

### **Cross-entropy loss**
For the label y, we can use the same representation as before. The only difference is that we now represent it with a probability vector, such as (0.1, 0.2, 0.7), rather than a vector containing only binary elements (0, 0, 1). We use [(3.4.8)](#348) to define the loss l, which is the expected loss value across all label distributions. This loss is called cross-entropy loss, and it is one of the most commonly used loss functions in classification problems

## **3.4.7 Fundamentals of Information Theory**
Information theory involves encoding, decoding, transmission, and the processing of information or data as concisely as possible.

### **Entropy**
**The central idea of information theory is to quantify the information content in data. In information theory, this value is called the entropy of distribution $P$.** It can be derived using the following equation:
##### **(3.4.11)**
$$H[P]=\sum\limits_{j}-P(j)logP(j).$$
**One of the fundamental theorems of information theory states that to encode data randomly sampled from a distribution $p$, we need at least $H[P]$ “nats” to encode it. A “nat” is equivalent to a bit, but with a base of e rather than 2. Therefore, one nat is $\frac{1}
{log(2)} ≈ 1.44$ bits.**

### **Amount of information**
Imagine we have a data stream that needs to be compressed. If we can easily predict the next piece of data, then that data is easy to compress. Why is that? Let’s take an extreme example: suppose every piece of data in the stream is exactly the same—that would be a very boring data stream. Since they’re always the same, we always know what the next piece of data will be. Therefore, to convey the contents of the data stream, we don’t need to transmit any information. In other words, the event “the next piece of data is xx” contains no information.<br><br>
However, if we cannot fully predict every event, we may sometimes feel “surprise.” Claude Shannon chose to use the **information
measure $log \frac{1}{P(j)} = −logP(j)$ to quantify this degree of surprise.** When observing an event $j$, we assign it a (subjective) probability $P(j)$. When we assign a lower probability to an event, our surprise is greater, and the information content of that event is also greater. **The entropy defined in [(3.4.11)](#3411) is the expected value of the information content when the assigned probabilities truly match the data generation process.**

### **A Fresh Look at Cross-Entropy**
If we think of entropy $H(P)$ as “the degree of surprise experienced by someone who knows the true probability,” then what is cross-entropy? Cross-entropy from $P$ to $Q$ is denoted as $H(P,Q)$. We can think of cross-entropy as **“the expected surprise experienced by an observer with a subjective probability of $Q$ when viewing data generated according to probability $P$.” Cross-entropy reaches its minimum when P = Q. In this case, the cross-entropy from P to $Q$ is $H(P,P) = H(P)$.**<br><br>
**In short, we can approach the cross-entropy classification objective from two perspectives: (i) maximizing the likelihood of the observed data; (ii) minimizing the surprise required to convey the labels.**

## **3.4.8 Model Prediction and Evaluation**
After training a softmax regression model, given any sample features, we can predict the probability of each output class. Typically, we use the class with the highest predicted probability as the output class. If the prediction matches the actual class (label), the prediction is correct. In the following experiments, we will use accuracy to evaluate the model’s performance. Accuracy is equal to the ratio of the number of correct predictions to the total number of predictions.

## **Summary**
* The softmax operation takes a vector and maps it to probabilities.
* Softmax regression is suitable for classification problems; it uses the probability distribution of the output classes generated by the softmax function.
* Cross-entropy is an excellent measure of the difference between two probability distributions; it quantifies the number of bits required for a given model to encode the data.

In [ ]:
##